<a href="https://colab.research.google.com/github/04pys/ml-system-labs/blob/main/notebooks/lab05_03_Softmax_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# softmax regression의 손실함수를 low-level로 구현

In [2]:
import torch
import torch.nn.functional as F

In [3]:
torch.manual_seed(1)

In [4]:
z = torch.FloatTensor([1, 2, 3]) # 소프트맥스 함수 적용하기 전 값

In [5]:
hypothesis = F.softmax(z)
print(hypothesis)

tensor([0.0900, 0.2447, 0.6652])


/tmp/ipykernel_528/558831656.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  hypothesis = F.softmax(z)


In [6]:
hypothesis.sum()

tensor(1.)

In [7]:
#임의의 3 x 5 크기의 행렬을 랜덤값으로 만듦
z = torch.rand(3, 5, requires_grad=True)

In [8]:
print(z)

tensor([[0.7576, 0.2793, 0.4031, 0.7347, 0.0293],
        [0.7999, 0.3971, 0.7544, 0.5695, 0.4388],
        [0.6387, 0.5247, 0.6826, 0.3051, 0.4635]], requires_grad=True)


In [10]:
# 각 sample마다 소프트맥스 함수를 적용해줘야 하므로, dim = 1(열 방향)
hypothesis = F.softmax(z, dim=1)
print(hypothesis)

tensor([[0.2645, 0.1639, 0.1855, 0.2585, 0.1277],
        [0.2430, 0.1624, 0.2322, 0.1930, 0.1694],
        [0.2226, 0.1986, 0.2326, 0.1594, 0.1868]], grad_fn=<SoftmaxBackward0>)


In [11]:
# 각 샘플에 대한 임의의 레이블(정답값)
y = torch.randint(5, (3,)).long()
print(y)

tensor([0, 2, 1])


In [12]:
# 모든 원소가 0의 값을 가진 3 × 5 텐서 생성
y_one_hot = torch.zeros_like(hypothesis)
y_one_hot.scatter_(1, y.unsqueeze(1), 1) # scatter의 각 인자별로 의미:
# 첫번째 1 -> dim=1방향으로
#두번째 y.unsqueeze(1) -> y를 3, 크기에서 3x1 크기, 즉 행 3개에 열 하나짜리 행렬로 바꾼다는 뜻. 참고: y.unsqueeze(0)은 y를 1x3 즉 행렬 하나에 열 3개짜리 행렬로 바꾼다는 뜻
#세번째 1 -> 1을 추가.

#정리해보면, dim=1 즉 열 방향으로 y.unsqueeze(1)에서 가리키는 인덱스에 1을 집어넣으란 뜻.

tensor([[1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.]])

In [13]:
print(y.unsqueeze(1))

tensor([[0],
        [2],
        [1]])


In [14]:
#손실함수
# cost(W) = -(1/n) * Σ(i=1 to n) Σ(j=1 to k) y_j^(i) log(p_j^(i))

In [15]:
cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()

In [16]:
print(cost)

tensor(1.4689, grad_fn=<MeanBackward0>)


In [ ]:
# high - level 구현부터는 나중에 하기

In [19]:
#소프트맥스 함수의 출력값을 로그함수의 입력값으로 사용
# Low level
torch.log(F.softmax(z, dim=1))

tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]], grad_fn=<LogBackward0>)

In [18]:
#두개를 합쳐서 F.log_softmax 라는 함수가 있다
# High level
F.log_softmax(z, dim=1)

tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]],
       grad_fn=<LogSoftmaxBackward0>)

In [20]:
# Low level
# 첫번째 수식
(y_one_hot * -torch.log(F.softmax(z, dim=1))).sum(dim=1).mean()


tensor(1.4689, grad_fn=<MeanBackward0>)

In [21]:
# 두번째 수식
(y_one_hot * - F.log_softmax(z, dim=1)).sum(dim=1).mean()

tensor(1.4689, grad_fn=<MeanBackward0>)

In [22]:
# High level
# 세번째 수식
#F.nll_loss 를 하면 원핫 인코딩된 벡터를 안쓰고 실제 값인 y를 그대로 써도 된다
F.nll_loss(F.log_softmax(z, dim=1), y)

tensor(1.4689, grad_fn=<NllLossBackward0>)

In [23]:
# 네번째 수식
#F.nll_loss 와 F.log_softmax 함수를 포함하고 있다
F.cross_entropy(z, y)

tensor(1.4689, grad_fn=<NllLossBackward0>)

In [25]:
import torch.nn as nn

In [26]:
# 클래스로 소프트맥스 구현(위에까지는 함수로 구현)
# 1단계: 클래스로 객체 생성
criterion = nn.CrossEntropyLoss()

# 2단계: 생성된 객체 사용
loss = criterion(z, y)
print(loss)


tensor(1.4689, grad_fn=<NllLossBackward0>)


In [27]:
#클래스를 사용하는 이유: 설정값을 객체에 저장해두고 나중에 사용할 수 있기 때문입니다.
criterion = nn.CrossEntropyLoss()

loss1 = criterion(z, y)
loss2 = criterion(z, y)

z2 = torch.rand(3, 5, requires_grad=True)
y2 = torch.randint(5,(3,)).long()
loss3 = criterion(z2, y)

print(loss1, loss2, loss3)


tensor(1.4689, grad_fn=<NllLossBackward0>) tensor(1.4689, grad_fn=<NllLossBackward0>) tensor(1.4555, grad_fn=<NllLossBackward0>)


In [28]:
# 평균 대신 합계를 구하는 설정
# 손실함수 객체를 한 번만 생성. 이제 호출할때는 무조건 criterion으로만 호출함.
criterion = nn.CrossEntropyLoss(reduction='sum')

# 같은 객체로 여러 번 계산 가능
loss1 = criterion(z, y)           # 첫 번째 계산
loss2 = criterion(z, y)           # 두 번째 계산

# 새로운 데이터가 있다면
z2 = torch.rand(3, 5, requires_grad=True)
y2 = torch.randint(5, (3,)).long()
loss3 = criterion(z2, y2)         # 새 데이터로 계산

In [29]:
# 함수 방식에서는 매번 설정을 반복해야 함
loss_sum1 = F.cross_entropy(z, y, reduction='sum')
loss_sum2 = F.cross_entropy(z2, y2, reduction='sum')  # 설정 반복

In [30]:
#nn.CrossEntropyLoss 클래스 또한 F.cross_entropy 함수처럼 비용함수에 softmax 함수를 포함하고 있다

In [32]:
#소프트맥스 회귀 구현하기

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [34]:
torch.manual_seed(1)

In [36]:
#8개의 샘플과 4개의 특성이 있다.
#y_train의 값을 보니 2,1,0 세개 있는걸로 보아 클래스가 3개짜리인 분류 문제인걸 알 수 있다
x_train = [[1, 2, 1, 1],
           [2, 1, 3, 2],
           [3, 1, 3, 4],
           [4, 1, 5, 5],
           [1, 7, 5, 5],
           [1, 2, 5, 6],
           [1, 6, 6, 6],
           [1, 7, 7, 7]]
y_train = [2, 2, 2, 1, 1, 1, 0, 0]
x_train = torch.FloatTensor(x_train)
y_train = torch.LongTensor(y_train)


In [37]:
print(x_train.shape)
print(y_train.shape)

torch.Size([8, 4])
torch.Size([8])


In [38]:
#최종 사용할 레이블은 y_train에서 원-핫 인코딩을 한 결과이어야 합니다.
#클래스의 개수는 3개이므로 y_train에 원-핫 인코딩한 결과는 8 × 3의 개수를 가져야 합니다.
y_one_hot = torch.zeros(8, 3)
y_one_hot.scatter_(1,y_train.unsqueeze(1),1)
print(y_one_hot)

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.]])


In [39]:
print(y_one_hot.shape)

torch.Size([8, 3])


In [40]:
#결과 레이블인 y_one_hot의 크기가 8x3이므로 XW의 결과가 8x3이 되어야 한다.
#X는 8x4 이므로 W는 4x3이어야 한다
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

optimizer = optim.SGD([W, b], lr=0.1)

In [41]:
#로우레벨 버전
nb_epochs = 1000
for epoch in range(nb_epochs +1):
  #가설
  hypothesis = F.softmax(x_train.matmul(W) + b, dim = 1)#dim=1인 이유는 샘플이 8개인데, 특성이 4개이므로 특성 방향(열방향)으로 연산을 수행하라는 뜻

  #손실함수
  cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()

  #경사하강법(cost로 가설 개선)
  optimizer.zero_grad()
  cost.backward()
  optimizer.step()

  # 100번마다 로그 출력
  if epoch % 100 == 0:
    print('Epoch {:4d}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, cost.item()
    ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.761050
Epoch  200/1000 Cost: 0.689991
Epoch  300/1000 Cost: 0.643229
Epoch  400/1000 Cost: 0.604117
Epoch  500/1000 Cost: 0.568255
Epoch  600/1000 Cost: 0.533922
Epoch  700/1000 Cost: 0.500291
Epoch  800/1000 Cost: 0.466908
Epoch  900/1000 Cost: 0.433507
Epoch 1000/1000 Cost: 0.399962


In [44]:
new_x = torch.FloatTensor([[1, 2, 1, 1]]) # 2로 예측함, 정답도 2. 학습 잘됨

prediction = F.softmax(new_x.matmul(W) + b, dim=1)
print(prediction)

tensor([[0.0013, 0.0309, 0.9679]], grad_fn=<SoftmaxBackward0>)


In [45]:
#하이레벨로 구현하기(함수)

In [46]:
# 모델 초기화
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros((1, 3), requires_grad=True)
# optimizer 설정
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs+1):
  #cost 계산
  z = F.softmax(x_train.matmul(W) + b, dim=1)
  cost = F.cross_entropy(z, y_train)

  #cost로 H(x) 개선
  optimizer.zero_grad()
  cost.backward()
  optimizer.step()

  # 100번마다 로그 출력
  if epoch % 100 == 0:
      print('Epoch {:4d}/{} Cost: {:.6f}'.format(
          epoch, nb_epochs, cost.item()
      ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.871612
Epoch  200/1000 Cost: 0.805532
Epoch  300/1000 Cost: 0.776397
Epoch  400/1000 Cost: 0.758719
Epoch  500/1000 Cost: 0.746085
Epoch  600/1000 Cost: 0.736327
Epoch  700/1000 Cost: 0.728517
Epoch  800/1000 Cost: 0.722142
Epoch  900/1000 Cost: 0.716860
Epoch 1000/1000 Cost: 0.712426


In [47]:
#nn.Module로 소프트맥스 구현하기
# 모델을 선언 및 초기화. 4개의 특성을 가지고 3개의 클래스로 분류. input_dim=4, output_dim=3.
model = nn.Linear(4, 3) # output_dim이 3이어야한다. 정답의 차원이 3이기 때문

In [48]:
optimizer = optim.SGD(model.parameters(), lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs+1):
  z = model(x_train)
  cost = F.cross_entropy(z, y_train)

  optimizer.zero_grad()
  cost.backward()
  optimizer.step()

 # 20번마다 로그 출력
  if epoch % 100 == 0:
      print('Epoch {:4d}/{} Cost: {:.6f}'.format(
          epoch, nb_epochs, cost.item()
      ))

Epoch    0/1000 Cost: 1.616785
Epoch  100/1000 Cost: 0.658891
Epoch  200/1000 Cost: 0.573443
Epoch  300/1000 Cost: 0.518151
Epoch  400/1000 Cost: 0.473265
Epoch  500/1000 Cost: 0.433516
Epoch  600/1000 Cost: 0.396563
Epoch  700/1000 Cost: 0.360914
Epoch  800/1000 Cost: 0.325392
Epoch  900/1000 Cost: 0.289178
Epoch 1000/1000 Cost: 0.254148


In [49]:
#클래스로 소프트맥스 회귀 구현하기

In [59]:
#3개의 클래스로 분류하는 문제. 3개의 클래스란 3가지 중 하나를 고른다는 뜻
class SoftmaxClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 3) # Output이 3!

    def forward(self, x):
        return self.linear(x)


In [61]:
model = SoftmaxClassifierModel()

optimizer = optim.SGD(model.parameters(), lr=0.1)


In [62]:
nb_epochs = 1000
for epoch in range(nb_epochs+1):
  prediction = model(x_train)
  cost = F.cross_entropy(prediction, y_train)

  optimizer.zero_grad()
  cost.backward()
  optimizer.step()

  # 20번마다 로그 출력
  if epoch % 100 == 0:
      print('Epoch {:4d}/{} Cost: {:.6f}'.format(
          epoch, nb_epochs, cost.item()
      ))

Epoch    0/1000 Cost: 1.366217
Epoch  100/1000 Cost: 0.722726
Epoch  200/1000 Cost: 0.637564
Epoch  300/1000 Cost: 0.578576
Epoch  400/1000 Cost: 0.527363
Epoch  500/1000 Cost: 0.479316
Epoch  600/1000 Cost: 0.432700
Epoch  700/1000 Cost: 0.386693
Epoch  800/1000 Cost: 0.340930
Epoch  900/1000 Cost: 0.295757
Epoch 1000/1000 Cost: 0.255350
